In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
# Import necesssary modules
import numpy as np
from keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Embedding
import tensorflow as tf
from keras.layers import Dropout
from tensorflow import keras
import pickle

In [19]:
# Load the saved model
model = keras.models.load_model('/content/drive/MyDrive/result_models/trained_model.h5')

with open('/content/drive/MyDrive/result_models/tokenizer_code.pickle', 'rb') as handle:
    tokenizer_code = pickle.load(handle)

with open('/content/drive/MyDrive/result_models/tokenizer_code.pickle', 'rb') as handle:
    tokenizer_summaries = pickle.load(handle)

In [29]:
max_code_length = 500  # Choose an appropriate value based on your data
max_summary_length = 300  # Choose an appropriate value based on your data

vocab_code_size = len(tokenizer_code.word_index) + 1
vocab_summary_size = len(tokenizer_summaries.word_index) + 1

embedding_dim = 128
lstm_units = 16

# Encoder
encoder_inputs = Input(shape=(max_code_length,))
encoder_embedding = Embedding(vocab_code_size, embedding_dim)(encoder_inputs)
encoder_lstm = LSTM(lstm_units, return_state=True)
encoder_dropout = Dropout(0.2) # Add a dropout layer with rate 0.2
_, encoder_state_h, encoder_state_c = encoder_lstm(encoder_embedding)
encoder_dropout_output = encoder_dropout(encoder_state_h) # Apply dropout to the output
encoder_states = [encoder_dropout_output, encoder_state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(vocab_summary_size, embedding_dim)(decoder_inputs)
decoder_lstm = LSTM(lstm_units, return_sequences=True, return_state=True)
decoder_dropout = Dropout(0.2) # Add a dropout layer with rate 0.2
decoder_dropout_output = decoder_dropout(decoder_embedding) # Apply dropout to the input
decoder_outputs, _, _ = decoder_lstm(decoder_dropout_output, initial_state=encoder_states)
decoder_dense = Dense(vocab_summary_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [35]:
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(lstm_units,))
decoder_state_input_c = Input(shape=(lstm_units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_embedding_inference = Embedding(vocab_summary_size, embedding_dim)(decoder_inputs)
decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_embedding_inference, initial_state=decoder_states_inputs
)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)

# Reverse-lookup token index to decode sequences back to something readable.
reverse_input_char_index = dict((i, char) for char, i in tokenizer_code.word_index.items())
reverse_target_char_index = dict((i, char) for char, i in tokenizer_summaries.word_index.items())

def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1))
    # Populate the first character of target sequence with the start character.
    # target_seq[0, 0] = tokenizer_summaries.word_index["<start>"]

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        # Check if the sampled_token_index is in the reverse_target_char_index dictionary
        if sampled_token_index in reverse_target_char_index:
            sampled_char = reverse_target_char_index[sampled_token_index]
            decoded_sentence += " " + sampled_char
        else:
            print(f"Sampled token index {sampled_token_index} not found in reverse_target_char_index dictionary.")

        # Exit condition: either hit max length or find the stop token.
        if sampled_char == "<end>" or len(decoded_sentence) > max_summary_length:
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update states
        states_value = [h, c]

    return decoded_sentence

# Input source code as a string
input_source_code = '''decoder_state_input_h = Input(shape=(lstm_units,))
decoder_state_input_c = Input(shape=(lstm_units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]'''  # Replace with your desired source code string

# Tokenize the input source code string
input_sequence = tokenizer_code.texts_to_sequences([input_source_code])

# Pad the tokenized input sequence
padded_input_sequence = pad_sequences(input_sequence, maxlen=max_code_length, padding='post')

# Call the decode_sequence function with the preprocessed input sequence
input_seq = padded_input_sequence[0:1]  # Extract the first sequence from the preprocessed input
generated_summary = decode_sequence(input_seq)
print("Generated summary:", generated_summary)


1/1 [==============================] - 0s 32ms/step
Generated summary:  check_correctness_channelwise(f): findall(haystack, findall(haystack, nodes_seen.add(new_node) 'spatial', demangle_name(name, vparts) updated'.format(name, (actual logfile=logfile) resource['url'].split('.')[(-1)]).lower() handle_error(typ, d['params'] cmd(state, internationalization 'default_item_status':
